# 基于 SAC 的 PMM 策略强化学习训练

本 notebook 将使用 TorchRL 框架中的 SAC 算法来训练 PMM 市商策略，实现自动优化交易参数。

## 训练目标

-   **策略优化**: 自动学习最优的价差、订单数量、时间参数等
-   **风险控制**: 在获得收益的同时控制仓位风险
-   **适应性**: 能够适应不同市场条件的参数调整
-   **鲁棒性**: 训练出在真实市场中稳定表现的策略


In [7]:
# 环境设置和依赖导入
import numpy as np
import torch
import warnings
import os
from torch import nn
from tensordict import TensorDict
from hftbacktest import BacktestAsset
from lib.rl_env import create_pmm_env
from tqdm.notebook import tqdm  # 添加进度条支持

# 设置警告过滤和随机种子
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

# 🔧 全局交易配置（统一定义）
# 费率配置（正确的费率）
MAKER_FEE_RATE = -0.00003     # Maker费率 -0.003% (负费率返佣)
TAKER_FEE_RATE = 0.0007       # Taker费率 +0.07% (正费率收费)

# 交易规格配置
TICK_SIZE = 0.001             # 最小价格变动单位
LOT_SIZE = 0.1                # 最小交易数量单位

# 数据配置
TRAINING_PAIR = 'xrpusdt'
START_DATE = 20250630

# 显示交易配置信息
print(f"📊 全局交易配置:")
print(f"   Maker费率: {MAKER_FEE_RATE*100:.4f}% (负费率返佣)")
print(f"   Taker费率: {TAKER_FEE_RATE*100:.4f}% (正费率收费)")
print(f"   最小价格单位: {TICK_SIZE}")
print(f"   最小交易单位: {LOT_SIZE}")

# 设置设备（增加MPS支持）
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 使用CUDA GPU加速")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🍎 使用Apple Silicon MPS加速")
else:
    device = torch.device("cpu")
    print("💻 使用CPU")

print(f"负费率做市商环境初始化完成，使用设备: {device}")

📊 全局交易配置:
   Maker费率: -0.0030% (负费率返佣)
   Taker费率: 0.0700% (正费率收费)
   最小价格单位: 0.001
   最小交易单位: 0.1
🍎 使用Apple Silicon MPS加速
负费率做市商环境初始化完成，使用设备: mps


In [8]:
# 📊 数据分片配置
print("📊 初始化数据分片系统...")

# 检查数据文件
data_file = f'data/output/{TRAINING_PAIR}_{START_DATE}.npz'
if not os.path.exists(data_file):
    raise FileNotFoundError(f"数据文件不存在: {data_file}")

print(f"✅ 数据文件: {os.path.basename(data_file)}")

# 创建数据切片
def create_data_slices(data_file, num_slices=5):
    slice_dir = 'data/slices'
    os.makedirs(slice_dir, exist_ok=True)
    
    slice_files = [f"{slice_dir}/{TRAINING_PAIR}_slice_{i}.npz" for i in range(num_slices)]
    
    if all(os.path.exists(f) for f in slice_files):
        print(f"✅ 数据切片已存在")
        return slice_files
    
    print(f"🔪 创建{num_slices}个数据切片...")
    with np.load(data_file) as original_data:
        total_length = len(original_data[list(original_data.keys())[0]])
        slice_length = total_length // num_slices
        
        # 使用tqdm显示切片进度
        with tqdm(range(num_slices), desc="创建数据切片") as pbar:
            for i in pbar:
                start_idx = i * slice_length
                end_idx = total_length if i == num_slices - 1 else (i + 1) * slice_length
                
                slice_data = {key: original_data[key][start_idx:end_idx] for key in original_data.keys()}
                np.savez_compressed(slice_files[i], **slice_data)
                pbar.set_postfix({'切片': f'{i+1}/{num_slices}'})
    
    return slice_files

data_slices = create_data_slices(data_file)
print(f"✅ 数据分片系统准备完成，共{len(data_slices)}个片段")

📊 初始化数据分片系统...
✅ 数据文件: xrpusdt_20250630.npz
✅ 数据切片已存在
✅ 数据分片系统准备完成，共5个片段


In [9]:
# 负费率做市商配置
print("🤖 配置高盈利导向做市商参数...")

# 基础配置
obs_dim = 10    # 观测维度
action_dim = 9  # 动作维度

# 高盈利导向做市商训练配置（牺牲交易量换取盈利质量）
TRAINING_CONFIG = {
    'episodes_per_slice': 60,       # 每片段训练轮数
    'steps_per_episode': 15,        # 每轮步数
    'checkpoint_dir': 'checkpoints',
    'volume_reward_weight': 0.10,   # 大幅降低交易量权重 0.25 -> 0.10
    'profit_reward_weight': 0.90,   # 大幅提升盈利权重 0.75 -> 0.90
    
    # 内存管理配置
    'memory_management': {
        'cleanup_frequency': 10,     # 每10轮清理一次内存
        'max_actions_cache': 1000,   # 最大动作缓存数量
        'batch_save_frequency': 20,  # 每20轮保存一次
    },
    
    # 早停配置
    'early_stopping': {
        'enabled': True,
        'patience': 40,              # 增加耐心值 30 -> 40（等待更好的收敛）
        'min_delta': 0.002,          # 提高改善阈值 0.001 -> 0.002（更严格的改善标准）
        'monitor_window': 15,        # 增加监控窗口 10 -> 15（更稳定的评估）
        'min_episodes': 60,          # 增加最少训练轮数 50 -> 60（确保充分训练）
    },
    
    # 高盈利导向配置
    'profitability_focus': {
        'min_profit_threshold': 0.005,    # 最小盈利阈值：0.5%
        'quality_over_quantity': True,     # 质量优于数量
        'risk_adjusted_reward': True,      # 风险调整奖励
        'spread_optimization': True,       # 价差优化
        'patience_reward_enabled': True,   # 耐心奖励（鼓励长期持仓）
    }
}

os.makedirs(TRAINING_CONFIG['checkpoint_dir'], exist_ok=True)

print(f"✅ 高盈利导向做市商配置完成")
print(f"   观测维度: {obs_dim}")
print(f"   动作维度: {action_dim}")
print(f"   Maker费率: {MAKER_FEE_RATE*100:.4f}% (负费率返佣)")
print(f"   Taker费率: {TAKER_FEE_RATE*100:.4f}% (正费率收费)")
print(f"   🎯 盈利权重: {TRAINING_CONFIG['profit_reward_weight']} (大幅提升)")
print(f"   📉 交易量权重: {TRAINING_CONFIG['volume_reward_weight']} (大幅降低)")
print(f"   ⏰ 耐心值: {TRAINING_CONFIG['early_stopping']['patience']}轮 (增加)")
print(f"   🎚️ 改善阈值: {TRAINING_CONFIG['early_stopping']['min_delta']} (提高)")
print(f"   📊 监控窗口: {TRAINING_CONFIG['early_stopping']['monitor_window']}轮 (增加)")
print(f"   💰 最小盈利阈值: {TRAINING_CONFIG['profitability_focus']['min_profit_threshold']*100:.1f}%")
print(f"   🎯 策略重点: 盈利质量 > 交易数量")
print(f"   ⏱️ 耐心奖励: {'启用' if TRAINING_CONFIG['profitability_focus']['patience_reward_enabled'] else '禁用'}")
print(f"   💡 负费率优势: Maker订单获得 {abs(MAKER_FEE_RATE)*100:.4f}% 返佣")

🤖 配置高盈利导向做市商参数...
✅ 高盈利导向做市商配置完成
   观测维度: 10
   动作维度: 9
   Maker费率: -0.0030% (负费率返佣)
   Taker费率: 0.0700% (正费率收费)
   🎯 盈利权重: 0.9 (大幅提升)
   📉 交易量权重: 0.1 (大幅降低)
   ⏰ 耐心值: 40轮 (增加)
   🎚️ 改善阈值: 0.002 (提高)
   📊 监控窗口: 15轮 (增加)
   💰 最小盈利阈值: 0.5%
   🎯 策略重点: 盈利质量 > 交易数量
   ⏱️ 耐心奖励: 启用
   💡 负费率优势: Maker订单获得 0.0030% 返佣


In [10]:
# 🔄 负费率做市商训练系统（内存优化版）
import gc
import json
import time
import psutil

# 全局变量 - 使用更小的缓存
episode_rewards = []
training_actions = []


class MemoryManager:
    """内存管理器"""
    
    def __init__(self, max_actions_cache=1000):
        self.max_actions_cache = max_actions_cache
        self.cleanup_count = 0
    
    def cleanup_actions_cache(self):
        """清理动作缓存"""
        global training_actions
        if len(training_actions) > self.max_actions_cache:
            # 只保留最近的动作记录
            training_actions = training_actions[-self.max_actions_cache:]
            self.cleanup_count += 1
            print(f"🧹 清理动作缓存，保留最近{self.max_actions_cache}条记录")
    
    def get_memory_status(self):
        """获取内存状态"""
        process = psutil.Process()
        memory = psutil.virtual_memory()
        return {
            'rss_mb': process.memory_info().rss / 1024 / 1024,
            'usage_percent': memory.percent,
            'available_mb': memory.available / 1024 / 1024
        }
    
    def should_force_cleanup(self):
        """检查是否需要强制清理"""
        status = self.get_memory_status()
        return status['usage_percent'] > 85 or status['rss_mb'] > 2000


class EarlyStopping:
    """早停机制类"""

    def __init__(self, patience=30, min_delta=0.001, monitor_window=10, min_episodes=50):
        self.patience = patience
        self.min_delta = min_delta
        self.monitor_window = monitor_window
        self.min_episodes = min_episodes
        self.best_avg_reward = float('-inf')
        self.patience_counter = 0
        self.stopped = False

    def should_stop(self, rewards):
        """检查是否应该早停"""
        if len(rewards) < self.min_episodes:
            return False

        if len(rewards) < self.monitor_window:
            return False

        # 计算最近窗口的平均奖励
        recent_avg = np.mean(rewards[-self.monitor_window:])

        # 检查是否有改善
        if recent_avg > self.best_avg_reward + self.min_delta:
            self.best_avg_reward = recent_avg
            self.patience_counter = 0
        else:
            self.patience_counter += 1

        # 检查是否达到耐心限制
        if self.patience_counter >= self.patience:
            self.stopped = True
            return True

        return False

    def get_status(self):
        """获取早停状态信息"""
        return {
            'best_reward': self.best_avg_reward,
            'patience_left': max(0, self.patience - self.patience_counter),
            'stopped': self.stopped
        }


def monitor_memory():
    """内存监控"""
    process = psutil.Process()
    memory = psutil.virtual_memory()
    return {
        'rss_mb': process.memory_info().rss / 1024 / 1024,
        'usage_percent': memory.percent
    }


def aggressive_cleanup():
    """激进内存清理"""
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.empty_cache()
        torch.mps.synchronize()


def generate_market_maker_action(device):
    """生成高盈利导向的做市商动作（更小的价差和止盈，限制最大挂单量）"""
    # 使用更小的张量并立即转换为所需类型
    with torch.no_grad():
        action = torch.zeros(9, device=device, dtype=torch.float32)

        # 修改后的参数范围 - 更小的价差，更小的止盈，限制最大挂单量
        action[0] = torch.rand(1, device=device) * 0.0001 + 0.0000   # 买单价差：0.005%-0.015% (0.5-1.5bps) - 更小的价差
        action[1] = torch.rand(1, device=device) * 0.0001 + 0.0000   # 卖单价差：0.005%-0.015% (0.5-1.5bps) - 更小的价差
        action[2] = torch.rand(1, device=device) * 3.0 + 1.0          # 订单刷新时间：1-4秒 - 保持不变
        action[3] = torch.rand(1, device=device) * 0.002 + 0.0005     # 价格偏差阈值：0.05%-0.25% (5-25bps) - 保持不变
        action[4] = torch.rand(1, device=device) * 40.0 + 10.0        # 挂单时间限制：10-50秒 - 保持不变
        action[5] = torch.rand(1, device=device) * 150.0 + 50.0       # 最大同时挂单数：50-200个 - 限制在200以下
        action[6] = torch.rand(1, device=device) * 0.00005 + 0.0000  # 止盈比例：0.005%-0.01% (0.5-1bps) - 更小的止盈
        action[7] = torch.rand(1, device=device) * 0.00005 + 0.0000  # 止损比例：0.005%-0.01% (0.5-1bps) - 更小的止损
        action[8] = torch.rand(1, device=device) * 2000.0 + 500.0     # 单笔订单数量：500-2500张 - 保持不变

        # 确保整数参数
        action[5] = torch.clamp(torch.round(action[5]), 50, 200)      # 最大挂单数限制在50-200之间
        action[8] = torch.clamp(torch.round(action[8]), 500, 2500)    # 订单数量限制在500-2500张之间

    return action


def safe_tensor_to_float(tensor):
    """安全地将张量转换为Python浮点数（支持MPS）"""
    if tensor.device.type in ['cuda', 'mps']:
        return float(tensor.cpu().item())
    return float(tensor.item())


def safe_tensor_to_numpy(tensor):
    """安全地将张量转换为NumPy数组（支持MPS）"""
    if tensor.device.type in ['cuda', 'mps']:
        return tensor.cpu().numpy()
    return tensor.numpy()


def calculate_market_maker_reward(base_reward, action, trading_volume=0.0):
    """计算高盈利导向的做市商奖励（重点关注盈利质量而非交易量）"""
    # 基础PnL奖励（更大权重给盈利）
    if base_reward < 0:
        # 亏损时严重惩罚
        profit_reward = base_reward * TRAINING_CONFIG['profit_reward_weight'] * 2.0
    else:
        # 盈利时给予更高奖励
        profit_reward = base_reward * TRAINING_CONFIG['profit_reward_weight'] * 1.2
    
    # 降低交易量奖励权重，更注重盈利质量
    volume_reward = 0.0
    if base_reward > 0:  # 只有盈利时才给交易量奖励
        volume_reward = trading_volume * TRAINING_CONFIG['volume_reward_weight'] * 0.00005  # 降低权重

    # 盈利质量奖励（新增）
    profitability_bonus = 0.0
    if base_reward > 0.01:  # 盈利超过1%时给额外奖励
        profitability_bonus = min(base_reward * 0.5, 0.05)  # 最高5%的额外奖励

    # 价差合理性奖励（鼓励适中的价差）
    spread_sum = safe_tensor_to_float(action[0]) + safe_tensor_to_float(action[1])
    optimal_spread_range = (0.0001, 0.0003)  # 1-3bps的理想价差范围（调整为更小的范围）
    if optimal_spread_range[0] <= spread_sum <= optimal_spread_range[1]:
        spread_reward = 0.01  # 价差在理想范围内
    else:
        spread_reward = max(0, 0.01 - abs(spread_sum - 0.0002) * 50)  # 偏离理想范围的惩罚

    # 风险控制奖励（鼓励合理的挂单数量）
    max_orders = safe_tensor_to_float(action[5])
    if 80 <= max_orders <= 150:  # 理想的挂单数量范围（调整为更高的范围）
        risk_control_reward = 0.005
    else:
        risk_control_reward = max(0, 0.005 - abs(max_orders - 115) * 0.00005)

    # 持仓时间奖励（鼓励更长的持仓时间以获得更好价格）
    hold_time = safe_tensor_to_float(action[4])
    if hold_time >= 20:  # 持仓时间超过20秒
        patience_reward = min(hold_time / 100, 0.008)  # 最高0.8%的耐心奖励
    else:
        patience_reward = 0

    total_reward = (profit_reward + volume_reward + profitability_bonus + 
                   spread_reward + risk_control_reward + patience_reward)
    
    return total_reward


def train_on_slice(slice_file, slice_idx, early_stopping=None, memory_manager=None):
    """训练单个数据片段（内存优化版）"""
    start_time = time.time()

    # 创建环境（使用全局常量）
    data_asset = (
        BacktestAsset()
        .data([slice_file])
        .linear_asset(1.0)
        .risk_adverse_queue_model()
        .no_partial_fill_exchange()
        .tick_size(TICK_SIZE)           # 使用全局常量
        .lot_size(LOT_SIZE)             # 使用全局常量
        .trading_value_fee_model(MAKER_FEE_RATE, TAKER_FEE_RATE)  # 使用全局常量
    )

    # 根据设备类型选择适当的设备参数
    device_str = device.type if device.type != 'mps' else 'cpu'

    env = create_pmm_env(
        data_asset=data_asset,
        initial_balance=10000.0,
        max_steps=25,  # 减少最大步数
        device=device_str,
        risk_penalty_weight=0.02,
        # 移除不现实的 transaction_cost_rate 参数
        step_interval_ns=50_000_000,
        order_amount_range=(1, 50),
    )

    slice_rewards = []
    local_actions = []  # 本地动作缓存

    # 使用tqdm显示episode进度
    with tqdm(range(TRAINING_CONFIG['episodes_per_slice']),
              desc=f"片段{slice_idx+1}",
              unit="episode") as pbar:
        for episode in pbar:
            episode_reward = 0
            episode_actions = []

            try:
                with torch.no_grad():  # 禁用梯度计算
                    tensordict = env.reset()

                    for step in range(TRAINING_CONFIG['steps_per_episode']):
                        action = generate_market_maker_action(device)
                        action_td = TensorDict(
                            {"action": action}, batch_size=(), device=device)
                        next_tensordict = env.step(action_td)

                        if 'next' in next_tensordict:
                            base_reward = safe_tensor_to_float(
                                next_tensordict['next']['reward'])
                            done = safe_tensor_to_float(
                                next_tensordict['next']['done']) > 0.5

                            estimated_volume = safe_tensor_to_float(
                                action[5]) * safe_tensor_to_float(action[8]) * 0.1
                            market_maker_reward = calculate_market_maker_reward(
                                base_reward, action, estimated_volume)

                            episode_reward += market_maker_reward

                            # 保存动作数据（修复：包含完整动作参数）
                            episode_actions.append({
                                'reward': market_maker_reward,
                                'base_reward': base_reward,
                                'volume': estimated_volume,
                                'action': safe_tensor_to_numpy(action).tolist()  # 保存完整动作参数
                            })

                            if done:
                                break

                            tensordict = TensorDict({
                                'observation': next_tensordict['next']['observation'],
                                'done': next_tensordict['next']['done']
                            }, batch_size=(), device=device)

            except Exception as e:
                episode_reward = -1.0

            slice_rewards.append(episode_reward)
            episode_rewards.append(episode_reward)

            # 保存动作到全局列表（修复：添加动作记录）
            for action_data in episode_actions:
                training_actions.append({
                    'slice_idx': slice_idx,
                    'episode': episode,
                    'reward': action_data['reward'],
                    'base_reward': action_data['base_reward'],
                    'volume': action_data['volume'],
                    'action': action_data['action']
                })

            # 内存管理
            if memory_manager:
                if episode % TRAINING_CONFIG['memory_management']['cleanup_frequency'] == 0:
                    aggressive_cleanup()
                    memory_manager.cleanup_actions_cache()
                
                if memory_manager.should_force_cleanup():
                    # print(f"⚠️ 强制内存清理触发")
                    aggressive_cleanup()

            # 更新进度条显示
            avg_reward = np.mean(
                slice_rewards[-10:]) if len(slice_rewards) >= 10 else np.mean(slice_rewards)

            # 早停检查
            postfix_info = {
                '平均奖励': f'{avg_reward:.4f}',
                '当前奖励': f'{episode_reward:.4f}'
            }

            if memory_manager:
                mem_status = memory_manager.get_memory_status()
                postfix_info['内存'] = f'{mem_status["rss_mb"]:.0f}MB'

            if early_stopping and early_stopping.should_stop(episode_rewards):
                status = early_stopping.get_status()
                postfix_info['早停'] = f'已触发'
                pbar.set_postfix(postfix_info)
                break
            elif early_stopping:
                status = early_stopping.get_status()
                postfix_info['耐心'] = f'{status["patience_left"]}'

            pbar.set_postfix(postfix_info)

    # 清理资源
    del env, data_asset
    aggressive_cleanup()

    return slice_rewards


def run_training():
    """运行负费率做市商训练（内存优化版）"""
    total_start_time = time.time()
    
    # 初始化内存管理器
    memory_manager = MemoryManager(
        max_actions_cache=TRAINING_CONFIG['memory_management']['max_actions_cache']
    )

    # 初始化早停机制
    early_stopping = None
    if TRAINING_CONFIG['early_stopping']['enabled']:
        early_stopping_params = {
            k: v for k, v in TRAINING_CONFIG['early_stopping'].items() if k != 'enabled'}
        early_stopping = EarlyStopping(**early_stopping_params)

    print(f"🚀 开始内存优化训练...")
    initial_mem = memory_manager.get_memory_status()
    print(f"📊 初始内存状态: {initial_mem['rss_mb']:.0f}MB ({initial_mem['usage_percent']:.1f}%)")

    # 使用tqdm显示整体训练进度
    with tqdm(enumerate(data_slices),
              total=len(data_slices),
              desc="训练进度",
              unit="片段") as pbar:
        for i, slice_file in pbar:
            pbar.set_description(f"训练片段 {i+1}/{len(data_slices)}")
            train_on_slice(slice_file, i, early_stopping, memory_manager)

            # 检查全局早停
            if early_stopping and early_stopping.stopped:
                print(f"\n🛑 早停触发！在片段 {i+1} 停止训练")
                status = early_stopping.get_status()
                print(f"   最佳平均奖励: {status['best_reward']:.4f}")
                print(f"   训练轮数: {len(episode_rewards)}")
                break

            # 片段间内存清理
            aggressive_cleanup()
            memory_manager.cleanup_actions_cache()

            # 更新整体进度显示
            avg_reward = np.mean(episode_rewards) if episode_rewards else 0
            mem_status = memory_manager.get_memory_status()
            postfix_info = {
                '总轮数': len(episode_rewards),
                '平均奖励': f'{avg_reward:.4f}',
                '内存': f'{mem_status["rss_mb"]:.0f}MB'
            }

            if early_stopping:
                status = early_stopping.get_status()
                postfix_info['耐心'] = f'{status["patience_left"]}'

            pbar.set_postfix(postfix_info)

    total_time = time.time() - total_start_time
    final_mem = memory_manager.get_memory_status()
    print(f"\n✅ 训练完成，用时: {total_time/60:.1f}分钟")
    print(f"💾 最终内存: {final_mem['rss_mb']:.0f}MB ({final_mem['usage_percent']:.1f}%)")

    if early_stopping and early_stopping.stopped:
        print(f"📊 早停统计: 最佳奖励 {early_stopping.best_avg_reward:.4f}")

    return episode_rewards


print("✅ 高盈利导向做市商训练系统准备完成（更小价差和止盈版本）")

✅ 高盈利导向做市商训练系统准备完成（更小价差和止盈版本）


In [11]:
# 🚀 执行训练
print("🚀 执行负费率做市商训练...")

# 检查系统状态
mem_info = monitor_memory()
print(
    f"📊 预计训练量: {len(data_slices)} × {TRAINING_CONFIG['episodes_per_slice']} = {len(data_slices) * TRAINING_CONFIG['episodes_per_slice']}轮")

# 执行训练
try:
    start_time = time.time()
    rewards = run_training()
    end_time = time.time()

    print(f"\n🎉 训练完成！")
    print(f"⏱️ 总用时: {(end_time - start_time)/60:.1f}分钟")

    if rewards:
        print(f"\n📊 训练统计:")
        print(f"   总轮数: {len(rewards)}")
        print(f"   平均奖励: {np.mean(rewards):.4f}")
        print(f"   最佳奖励: {max(rewards):.4f}")
        print(
            f"   成功率: {len([r for r in rewards if r > 0])/len(rewards)*100:.1f}%")

        # 最佳策略
        if training_actions:
            best_action = max(training_actions, key=lambda x: x['reward'])
            action_params = best_action['action']

            print(f"\n🏆 最佳策略 (奖励: {best_action['reward']:.4f}):")
            print(f"   买入价差: {action_params[0]*10000:.1f}bps")
            print(f"   卖出价差: {action_params[1]*10000:.1f}bps")
            print(f"   刷新时间: {action_params[2]:.2f}s")
            print(f"   最大挂单: {int(action_params[5])}个")
            print(f"   订单数量: {int(action_params[8])}张")

            # 收益预估
            daily_volume = int(
                action_params[5]) * int(action_params[8]) * 1440 / (action_params[2] + action_params[4])
            fee_income = daily_volume * 0.0002
            print(f"\n💰 收益预估:")
            print(f"   日交易量: {daily_volume:.0f}张")
            print(f"   日手续费收入: {fee_income:.2f} USDT")

except Exception as e:
    print(f"❌ 训练失败: {e}")

finally:
    aggressive_cleanup()  # 修复：使用正确的函数名
    final_mem = monitor_memory()
    print(f"\n💾 最终内存: {final_mem['rss_mb']:.0f}MB")

🚀 执行负费率做市商训练...
📊 预计训练量: 5 × 60 = 300轮
🚀 开始内存优化训练...
📊 初始内存状态: 140MB (82.1%)


训练进度:   0%|          | 0/5 [00:00<?, ?片段/s]

片段1:   0%|          | 0/60 [00:00<?, ?episode/s]

片段2:   0%|          | 0/60 [00:00<?, ?episode/s]


🛑 早停触发！在片段 2 停止训练
   最佳平均奖励: -3.3474
   训练轮数: 117

✅ 训练完成，用时: 0.9分钟
💾 最终内存: 1152MB (83.0%)
📊 早停统计: 最佳奖励 -3.3474

🎉 训练完成！
⏱️ 总用时: 0.9分钟

📊 训练统计:
   总轮数: 117
   平均奖励: -5.2064
   最佳奖励: 1.2888
   成功率: 0.9%

🏆 最佳策略 (奖励: 1.2888):
   买入价差: 0.5bps
   卖出价差: 0.6bps
   刷新时间: 2.05s
   最大挂单: 134个
   订单数量: 840张

💰 收益预估:
   日交易量: 4570416张
   日手续费收入: 914.08 USDT

💾 最终内存: 1152MB


In [12]:
# 📊 结果分析和参数保存
print("📊 分析训练结果...")


def safe_convert_rewards(rewards):
    """安全转换奖励列表（处理可能的张量）"""
    converted_rewards = []
    for reward in rewards:
        if isinstance(reward, torch.Tensor):
            if reward.device.type in ['cuda', 'mps']:
                converted_rewards.append(float(reward.cpu().item()))
            else:
                converted_rewards.append(float(reward.item()))
        else:
            converted_rewards.append(float(reward))
    return converted_rewards


def save_optimal_strategy_json(optimal_config, filename='optimal_strategy_config.json'):
    """保存最优策略配置到JSON文件"""
    import time

    config_data = {
        "metadata": {
            "training_date": time.strftime('%Y-%m-%d %H:%M:%S'),
            "training_pair": TRAINING_PAIR.upper(),
            "maker_fee_rate": MAKER_FEE_RATE,
            "taker_fee_rate": TAKER_FEE_RATE,
            "training_episodes": len(episode_rewards),
            "average_reward": np.mean(safe_convert_rewards(episode_rewards)),
            "best_reward": max(safe_convert_rewards(episode_rewards)),
            "success_rate": len([r for r in safe_convert_rewards(episode_rewards) if r > 0])/len(episode_rewards)*100
    },
        "optimal_config": optimal_config,
    }

    with open(f'checkpoints/{filename}', 'w', encoding='utf-8') as f:
        json.dump(config_data, f, indent=4, ensure_ascii=False)

    print(f"✅ 最优策略配置已保存到: checkpoints/{filename}")


def analyze_results():
    """分析训练结果"""
    if not episode_rewards:
        print("❌ 无训练数据")
        return None

    # 安全转换奖励
    safe_rewards = safe_convert_rewards(episode_rewards)

    print(f"📈 训练统计:")
    print(f"   总轮数: {len(safe_rewards)}")
    print(f"   平均奖励: {np.mean(safe_rewards):.4f}")
    print(f"   标准差: {np.std(safe_rewards):.4f}")
    print(f"   最大值: {max(safe_rewards):.4f}")
    print(f"   最小值: {min(safe_rewards):.4f}")

    # 盈利性分析
    positive_rewards = [r for r in safe_rewards if r > 0]
    if positive_rewards:
        profitability = len(positive_rewards) / len(safe_rewards) * 100
        print(f"\n💰 盈利性分析:")
        print(f"   盈利比例: {profitability:.1f}%")
        print(f"   平均盈利: {np.mean(positive_rewards):.4f}")

    # 最佳策略配置
    optimal_config = None
    if training_actions:
        sorted_actions = sorted(
            training_actions, key=lambda x: x['reward'], reverse=True)
        top_10_percent = sorted_actions[:max(1, len(sorted_actions)//10)]

        if top_10_percent:
            print(f"\n🏆 最优策略配置 (前{len(top_10_percent)}个):")

            actions_array = np.array([action['action']
                                     for action in top_10_percent])
            avg_params = np.mean(actions_array, axis=0)

            optimal_config = {
                "bid_spread": float(avg_params[0]),
                "ask_spread": float(avg_params[1]),
                "order_refresh_time": float(avg_params[2]),
                "price_deviation_pct": float(avg_params[3]),
                "hang_order_time_limit": float(avg_params[4]),
                "max_open_orders": int(avg_params[5]),
                "take_profit_pct": float(avg_params[6]),
                "stop_loss_pct": float(avg_params[7]),
                "order_amount": int(avg_params[8])
            }

            print(
                f"   买入价差: {optimal_config['bid_spread']:.6f} ({optimal_config['bid_spread']*10000:.1f}bps)")
            print(
                f"   卖出价差: {optimal_config['ask_spread']:.6f} ({optimal_config['ask_spread']*10000:.1f}bps)")
            print(f"   刷新时间: {optimal_config['order_refresh_time']:.2f}s")
            print(
                f"   价格偏差: {optimal_config['price_deviation_pct']:.6f} ({optimal_config['price_deviation_pct']*10000:.1f}bps)")
            print(f"   挂单时限: {optimal_config['hang_order_time_limit']:.1f}s")
            print(f"   最大挂单: {optimal_config['max_open_orders']}个")
            print(
                f"   止盈比例: {optimal_config['take_profit_pct']:.6f} ({optimal_config['take_profit_pct']*10000:.1f}bps)")
            print(
                f"   止损比例: {optimal_config['stop_loss_pct']:.6f} ({optimal_config['stop_loss_pct']*10000:.1f}bps)")
            print(f"   订单数量: {optimal_config['order_amount']}张")

            # 收益预估（使用全局常量）
            daily_volume = optimal_config['max_open_orders'] * optimal_config['order_amount'] * 1440 / (
                optimal_config['order_refresh_time'] + optimal_config['hang_order_time_limit'])
            fee_income = daily_volume * abs(MAKER_FEE_RATE)  # 使用全局常量

            print(f"\n💰 负费率做市商收益预估:")
            print(f"   日交易量: {daily_volume:.0f}张")
            print(f"   日手续费收入: {fee_income:.2f} USDT")
            print(f"   月收入: {fee_income * 30:.0f} USDT")
            print(f"   年收入: {fee_income * 365:.0f} USDT")

            # 生成用于03_strategy_design.ipynb的代码
            print(f"\n📋 用于 03_strategy_design.ipynb 的代码:")
            print(f"```python")
            print(f"_ = pmm.pmm_strategy(")
            print(f"    hbt, recorder.recorder,")
            print(f"    interval_seconds=0.1,")
            print(f"    bid_spread={optimal_config['bid_spread']:.6f},")
            print(f"    ask_spread={optimal_config['ask_spread']:.6f},")
            print(
                f"    order_refresh_time={optimal_config['order_refresh_time']:.2f},")
            print(
                f"    price_deviation_pct={optimal_config['price_deviation_pct']:.6f},")
            print(
                f"    hang_order_time_limit={optimal_config['hang_order_time_limit']:.1f},")
            print(f"    max_open_orders={optimal_config['max_open_orders']},")
            print(
                f"    take_profit_pct={optimal_config['take_profit_pct']:.6f},")
            print(f"    stop_loss_pct={optimal_config['stop_loss_pct']:.6f},")
            print(f"    order_amount={optimal_config['order_amount']},")
            print(f")")
            print(f"```")

    return optimal_config


def generate_summary():
    """生成训练总结"""
    if not episode_rewards:
        return

    safe_rewards = safe_convert_rewards(episode_rewards)

    print(f"\n📋 负费率做市商训练总结:")
    print(f"=" * 60)
    print(f"🎯 训练模式: 负费率做市商 (Maker {MAKER_FEE_RATE*100:.3f}%)")  # 使用全局常量
    print(f"📊 训练规模: {len(data_slices)}个数据片段")
    print(f"🔢 总训练轮数: {len(safe_rewards)}")
    print(f"📈 动作记录数: {len(training_actions)}")
    print(f"💰 训练对象: {TRAINING_PAIR.upper()}")
    print(f"📅 训练时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")

    if safe_rewards:
        success_rate = len([r for r in safe_rewards if r > 0]
                           ) / len(safe_rewards) * 100
        print(f"✅ 成功率: {success_rate:.1f}%")
        print(f"📊 平均表现: {np.mean(safe_rewards):.4f}")
        print(f"🏆 最佳表现: {max(safe_rewards):.4f}")

    print(f"🎉 负费率做市商训练完成!")


# 执行分析
optimal_config = analyze_results()
generate_summary()

# 保存最优策略配置到JSON
if optimal_config:
    filename = f'{TRAINING_PAIR}_optimal_config_{time.strftime("%Y%m%d_%H%M%S")}.json'
    save_optimal_strategy_json(optimal_config, filename)
else:
    print("⚠️ 未找到可用的训练动作数据，无法保存配置")

📊 分析训练结果...
📈 训练统计:
   总轮数: 117
   平均奖励: -5.2064
   标准差: 2.6698
   最大值: 1.2888
   最小值: -17.9868

💰 盈利性分析:
   盈利比例: 0.9%
   平均盈利: 1.2888

🏆 最优策略配置 (前11个):
   买入价差: 0.000046 (0.5bps)
   卖出价差: 0.000048 (0.5bps)
   刷新时间: 2.53s
   价格偏差: 0.001483 (14.8bps)
   挂单时限: 22.3s
   最大挂单: 124个
   止盈比例: 0.000024 (0.2bps)
   止损比例: 0.000018 (0.2bps)
   订单数量: 1610张

💰 负费率做市商收益预估:
   日交易量: 11569573张
   日手续费收入: 347.09 USDT
   月收入: 10413 USDT
   年收入: 126687 USDT

📋 用于 03_strategy_design.ipynb 的代码:
```python
_ = pmm.pmm_strategy(
    hbt, recorder.recorder,
    interval_seconds=0.1,
    bid_spread=0.000046,
    ask_spread=0.000048,
    order_refresh_time=2.53,
    price_deviation_pct=0.001483,
    hang_order_time_limit=22.3,
    max_open_orders=124,
    take_profit_pct=0.000024,
    stop_loss_pct=0.000018,
    order_amount=1610,
)
```

📋 负费率做市商训练总结:
🎯 训练模式: 负费率做市商 (Maker -0.003%)
📊 训练规模: 5个数据片段
🔢 总训练轮数: 117
📈 动作记录数: 117
💰 训练对象: XRPUSDT
📅 训练时间: 2025-07-01 19:24:57
✅ 成功率: 0.9%
📊 平均表现: -5.2064
🏆 最佳表现: 1.2888
🎉 

: 